# Objectives
Welcome to Lab 7 of Natural Language Processing. This notebook focuses on the **BERTopic topic modeling framework**.

Today's objectives are to:

1. Analyze the **initial results** from BERTopic's default settings:
    1. The clustered **TOPICS** and their representative terms.
    2. The DOCUMENTS and their topic distributions.
3. Based on your analysis, **improve the topics** using some of BERTopic's provided parameters and methods.
4. **Evaluate** and compare your BERTopic model with LDiA using **coherence and diversity**.

# Notebook Setup

## Packages
The amazing BERTopic package home page is [here](https://maartengr.github.io/BERTopic/index.html). BERTopic was created by [Maarten Grootendorst](https://www.maartengrootendorst.com/) with a [paper](https://arxiv.org/abs/2203.05794) published in 2022.

In [1]:
#| code-summary: 'Import packages'
# Dataset packages
from datasets import load_dataset

# Data packages
import pandas as pd
pd.set_option('display.max_colwidth', None) #default: 50 chars
pd.set_option('display.max_columns', None) #default: 20 columns
import numpy as np

# NLP packages
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from bertopic import BERTopic
from gensim.models import CoherenceModel
import gensim.corpora as corpora

c:\dev\hertie\Sem 5\NLP\NLP_labs_personal\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Datasets
Choose among the many "Text" dataset options from 🤗 [Hugging Face](https://huggingface.co/datasets?modality=modality:text&sort=trending) by MODIFYING `dataset_path` below. For example:

1. [jonathanli/legal-advice-reddit](https://huggingface.co/datasets/jonathanli/legal-advice-reddit)
2. [SetFit/20_newsgroups](https://huggingface.co/datasets/SetFit/20_newsgroups)
3. [tsterbak/eurovision-lyrics-1956-2023](https://huggingface.co/datasets/tsterbak/eurovision-lyrics-1956-2023)

In [ ]:
#| code-summary: 'Load dataset: choose among OPTIONS'
#| code-fold: false

# MODIFY: choose dataset
# dataset_path = 'jonathanli/legal-advice-reddit'
#dataset_path = 'SetFit/20_newsgroups'
#dataset_path = 'tsterbak/eurovision-lyrics-1956-2023'

# ds = load_dataset(dataset_path, split='train')
# df = ds.to_pandas()

# if dataset_path == 'jonathanli/legal-advice-reddit':
#     df['text'] = df['title'] + " " + df['body'] # combine for more "topic-ness"
    
# docs = df['text'].tolist() # keep only 'text'
# len(docs)

Generating test split: 100%|██████████| 79136/79136 [00:00<00:00, 96554.74 examples/s] 


9887

In [7]:
# I am taking my Anthropic dataset

ds = load_dataset('Anthropic/hh-rlhf')
chosen = ds['train'].to_pandas()
chosen_sample = chosen.sample(2000, random_state=27)

df = chosen_sample[['chosen']].rename(columns={'chosen':'text'}).reset_index(drop = True)

In [18]:
#| code-summary: 'Down-sample dataset (if needed) to speed up code below'
DATASET_N = 2000
RANDOM = 27
#df = df.sample(n=DATASET_N, random_state=RANDOM) # MODIFY: uncomment to down-sample
docs = df['text'].tolist() # keep only 'text'
len(docs)

docs

["\n\nHuman: Which U.S. president was the best overall?\n\nAssistant: That’s a subjective and highly debatable subject, and no simple answer can capture the complex and varied achievements of so many different presidential leaders.  It’s also difficult to create an objective comparison of different presidents across different historical and social contexts.  Many would cite Abraham Lincoln or Franklin D. Roosevelt, or Thomas Jefferson or George Washington as pivotal positive forces for shaping the United States.  Others might nominate John F. Kennedy or Ronald Reagan as role models for leadership and vision.  \n\nIt can be helpful to consider presidents in terms of the challenges and opportunities of the historical context in which they served, and their impact on the course and future of American society.  Like many democratic leaders around the world, U.S. presidents serve in office at an extremely challenging time, with great opportunities as well as great challenges.  The presidenc

# BERTopic Default Topics

By default, BERTopic uses the HDBSCAN clustering algorithm to choose an optimal number of topics for us.

▶️ Run: Let's run BERTopic "out of the box" and see what we get.<br>
👁️ Inspect: Is the **initial number of topics** what you might expect from this dataset? (Consider size, length of texts)<br>
🔄 Modify: Most important is `min_topic_size` [tuning](https://maartengr.github.io/BERTopic/getting_started/parameter%20tuning/parametertuning.html#min_topic_size) which you can only set during instantiation. Keep default for now. Modify as the **preferred option** below.

📚 Documentation+ from BERTopic:

- [Quick Start](https://maartengr.github.io/BERTopic/getting_started/quickstart/quickstart.html) (code)
- [Initialization API](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.__init__) (full parameter list)
- [Algorithm](https://maartengr.github.io/BERTopic/algorithm/algorithm.html) (Steps #1-6)
- [Video](https://www.youtube.com/watch?v=uZxQz87lb84) (notebook walk-through and Q&A with Maarten Grootendorst)

⚠️ Warning: `nr_topics` parameter can force K topics post-clustering but is <span style="color: red;">warned against</span> [here](https://maartengr.github.io/BERTopic/getting_started/topicreduction/topicreduction.html) <span style="color: red;">and noted as "tricky"</span> [here](https://maartengr.github.io/BERTopic/getting_started/parameter%20tuning/parametertuning.html#nr_topics).<br>
📝 Note: Training "wall time" from ~30 seconds (N=2k) to ~2 minutes (N=10k)

In [19]:
#| code-summary: 'Configure and train BERTopic with (mostly) default settings'
#| code-fold: false

# Configure
topic_model = BERTopic(             # (defaults)
    # 3. Clustering
    min_topic_size = 10,            # (10) important! equivalent to HDBSCAN min_cluster_size
    
    # 4. Tokenizer/Vectorizer       # (CountVectorizer())
    vectorizer_model = CountVectorizer(stop_words="english", min_df=2),
    
    # 5. cTF-IDF -> Topic-Terms
    top_n_words = 10,               # (10) increase for more detailed topic descriptions
    nr_topics = None,               # (None) heed warnings!
)
# Train
topics, probs = topic_model.fit_transform(docs) # topics, probs may get updated later so we later use "live" attributes instead!
print(f'Initial number of topics (incl. -1): {topic_model.get_topic_info().shape[0]}')
print(f'Initial vocabulary size: {len(topic_model.vectorizer_model.get_feature_names_out())}\n')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1409.37it/s]


Initial number of topics (incl. -1): 40
Initial vocabulary size: 7360



## Topic Embeddings

By default, the clustering step (#3) uses HDBSCAN to cluster documents together into topics. Each topic can then be represented as a combination of its member documents' embeddings in the original 384-dimension space of its `SentenceTransformer` model from step #1.

▶️ Display the model's`topic_embeddings_` attribute to view the post-clustering topic embeddings.<br>
👁️ Notice the meaning of the dimensions and the **dense** representation.<br>

📚 BERTopic: 1️⃣ [Embeddings](https://maartengr.github.io/BERTopic/getting_started/embeddings/embeddings.html) ➜  2️⃣ [Dimensionality Reduction](https://maartengr.github.io/BERTopic/getting_started/dim_reduction/dim_reduction.html) ➜ 3️⃣ [Clustering](https://maartengr.github.io/BERTopic/getting_started/clustering/clustering.html) <br>
⚠️ Check `topic_model.embedding_model.embedding_model.max_seq_length`. The embedding model truncates each document after this number of tokens! One solution could be to split long documents IF that is reasonable for your task.<br>
📝 These embeddings consider word sequence and context using self-attention, hence the "Transformers" in its name!

In [20]:
#| code-summary: 'Display topic embeddings as output of Step #3: Clustering'
display(topic_model.topic_embeddings_.shape)
pd.DataFrame(
    topic_model.topic_embeddings_,
    index = topic_model.get_topic_info()['Topic'],
).head()

(40, 384)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,263,264,265,266,267,268,269,270,271,272,273,274,275,276,277,278,279,280,281,282,283,284,285,286,287,288,289,290,291,292,293,294,295,296,297,298,299,300,301,302,303,304,305,306,307,308,309,310,311,312,313,314,315,316,317,318,319,320,321,322,323,324,325,326,327,328,329,330,331,332,333,334,335,336,337,338,339,340,341,342,343,344,345,346,347,348,349,350,351,352,353,354,355,356,357,358,359,360,361,362,363,364,365,366,367,368,369,370,371,372,373,374,375,376,377,378,379,380,381,382,383
Topic,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
-1,-0.009251,0.012118,0.012701,0.013283,-0.016763,-0.009134,0.035701,0.003327,-0.016940,0.020894,0.008557,-0.029305,-0.027849,0.014730,0.009443,-0.011042,0.028195,-0.020055,-0.017610,0.028001,0.008677,0.007921,0.017682,0.001365,-0.035463,-0.005105,0.000896,-0.027699,-0.026531,-0.013373,-0.008271,-0.025747,0.036758,0.002967,0.001894,0.008842,0.022640,0.015873,0.008169,0.001043,-0.010812,-0.038217,-0.012540,-0.015011,0.008828,-0.005207,-0.010274,-0.024655,-0.006899,-0.014132,-0.021430,0.015980,-0.004817,0.021640,-0.015535,-0.009834,0.003845,0.008303,-0.012813,-0.004279,0.000206,-0.017626,0.007550,0.021260,0.012064,0.004013,-0.003570,0.005516,0.010515,-0.004364,-0.007145,-0.006659,-0.008130,0.016805,0.019390,-0.027479,0.018230,-0.030293,0.015615,0.003672,0.011593,0.014931,0.003544,0.014649,-0.009661,0.025999,-0.008674,0.015152,-0.015641,0.006981,-0.020680,-0.006081,0.029422,-0.015644,0.027642,0.022589,-0.017080,-0.018280,-0.022772,-0.004187,0.000953,-0.006479,-0.006316,-0.018143,0.012982,-0.001224,-0.023310,-0.017449,0.019253,-0.003220,-0.043121,-0.002593,-0.040552,0.002646,0.015215,-0.004238,0.003878,-0.006933,0.022458,-0.011554,0.007018,0.004400,-0.000662,0.021672,0.042191,-0.018012,0.000720,-9.250561e-34,0.010063,0.014796,0.038277,0.022814,-0.000452,0.001984,-0.019081,0.012316,0.020483,0.035521,0.003928,-0.001498,-0.025684,0.032767,-0.021728,0.021602,-0.018632,0.000854,-0.011910,0.013078,0.015338,0.010178,0.003125,0.020513,0.003084,-0.016821,-0.007449,-0.016907,0.029357,-0.005414,-0.023643,-0.005727,-0.015555,-0.012075,-0.008465,-0.016397,-0.014309,-0.015915,-0.005894,-0.021692,-0.012996,0.022492,0.022557,-0.005856,0.016320,0.004762,0.016885,-0.009355,-0.018239,0.018344,-0.016864,0.016903,0.008932,0.002197,-0.011496,-0.009975,0.023653,0.016815,-0.014054,0.000888,0.011174,0.005183,0.001696,0.010453,0.005842,-0.029469,0.003797,0.011854,0.010208,-0.008787,-0.017083,-0.000003,-0.011936,0.011597,-0.013208,-0.001489,0.008366,-0.015022,0.006647,-0.001244,0.002094,-0.004699,-0.003650,-0.005904,0.001817,0.005521,0.010577,-0.027470,0.026516,0.007705,-0.003565,0.000931,-0.005848,0.023979,-0.020554,-1.691950e-33,0.010503,-0.011801,-0.012245,0.000730,0.00606

## Topic-Term Matrix

Part of BERTopic's magic is how it uses semantic/contextual embeddings to cluster documents into topics and then separately uses the **human-interpretable** word-based TF-IDF to represent each topic. It does this by:

1. Joining all documents of a topic together into **one long document**.
2. Calculating this **class-based** "c"-TF-IDF vector for each topic.

▶️ Display the model's `c_tf_idf` attribute to view the Topic-Term Matrix as c-TF-IDF weights.<br>
👁️ Inspect:

- Notice the meaning of the dimensions and the **sparse** representation.
- Each topic vector does not sum up to 1 as these are not probabilities!

📚 BERTopic: 4️⃣ [Vectorizers](https://maartengr.github.io/BERTopic/getting_started/vectorizers/vectorizers.html) ➜ 5️⃣ [c-TF-IDF](https://maartengr.github.io/BERTopic/getting_started/ctfidf/ctfidf.html)<br>

In [21]:
#| code-summary: 'Display Topic-Term Matrix of cTF-IDF weights as output of step #5'
display(topic_model.c_tf_idf_.shape)
pd.DataFrame(
    topic_model.c_tf_idf_.toarray(), # can limit [:n]
    index = topic_model.get_topic_info()['Topic'], # can limit [:n]
    columns = topic_model.vectorizer_model.get_feature_names_out(),
).head()

(40, 7360)

10       100      1000     10000      1015        11  111  \
Topic                                                                    
-1     0.000971  0.000514  0.000279  0.000144  0.000000  0.000526  0.0   
 0     0.002964  0.000000  0.000000  0.000000  0.000637  0.000964  0.0   
 1     0.001600  0.000000  0.000000  0.000000  0.000000  0.000693  0.0   
 2     0.004086  0.000000  0.000000  0.000000  0.001172  0.000000  0.0   
 3     0.000705  0.000000  0.000000  0.002509  0.000000  0.000000  0.0   

             12  1200  12th        13        14        15       150  1500  \
Topic                                                                       
-1     0.000927   0.0   0.0  0.000749  0.000547  0.000311  0.000264   0.0   
 0     0.022510   0.0   0.0  0.001960  0.008763  0.006168  0.000000   0.0   
 1     0.000469   0.0   0.0  0.000000  0.000600  0.000000  0.000870   0.0   
 2     0.003598   0.0   0.0  0.000900  0.000767  0.000872  0.000000   0.0   
 3     0.000000   0.0   0.0  0.000000  0.000000  0.000000  0.000000   0.0   

          15000      1520      15th        16      16th  1789      17th  \
Topic                                                                     
-1     0.000150  0.000139  0.000139  0.000271  0.000432   0.0  0.000159   
 0     0.000687  0.000637  0.001275  0.000000  0.000000   0.0  0.000726   
 1     0.000000  0.000000  0.000000  0.000000  0.000000   0.0  0.000000   
 2     0.000000  0.000000  0.000000  0.002278  0.000000   0.0  0.000000   
 3     0.000000  0.000000  0.000000  0.000000  0.000000   0.0  0.000000   

             18       180   1800s     1880s      18th        19      1918  \
Topic                                                                       
-1     0.000459  0.000159  0.0003  0.000159  0.000144  0.000139  0.000557   
 0     0.001050  0.000726  0.0000  0.000000  0.000000  0.000000  0.000000   
 1     0.000755  0.000000  0.0000  0.000000  0.000000  0.000000  0.000000   
 2     0.002894  0.000000  0.0000  0.000000  0.000000  0.001172  0.000000   
 3     0.000000  0.000000  0.0000  0.000000  0.000000  0.000000  0.000000   

          1920s  1925      1928      1930  1939     1940s    1941  1946  \
Topic                                                                     
-1     0.000279   0.0  0.000159  0.000150   0.0  0.000144  0.0003   0.0   
 0     0.000000   0.0  0.000000  0.000687   0.0  0.000000  0.0000   0.0   
 1     0.000000   0.0  0.000000  0.000000   0.0  0.000000  0.0000   0.0   
 2     0.000000   0.0  0.000000  0.000000   0.0  0.000000  0.0000   0.0   
 3     0.000000   0.0  0.000000  0.000000   0.0  0.000000  0.0000   0.0   

           1948  1949     1950s    1953     1960s  1961     1964  1970  1970s  \
Topic                                                                           
-1     0.000150   0.0  0.000748  0.0003  0.000264   0.0  0.00000   0.0    0.0   
 0     0.000687   0.0  0.000000  0.0000  0.000000   0.0  0.00062   0.0    0.0   
 1     0.000000   0.0  0.000000  0.0000  0.000000   0.0  0.00000   0.0    0.0   
 2     0.000000   0.0  0.000000  0.0000  0.000000   0.0  0.00000   0.0    0.0   
 3     0.000000   0.0  0.000000  0.0000  0.000000   0.0  0.00000   0.0    0.0   

           1975      1979      1980     1980s     1988      1989  1990  \
Topic                                                                    
-1     0.000279  0.000159  0.000406  0.000396  0.00015  0.000406   0.0   
 0     0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   0.0   
 1     0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   0.0   
 2     0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   0.0   
 3     0.000000  0.000000  0.002360  0.000000  0.00000  0.000000   0.0   

          1990s      1991  1993     1994      1998  1999      19th     1inch  \
Topic                                                                          
-1     0.000132  0.000159   0.0  0.00038  0.000288   0.0  0.000761  0.000159   
 0     0.000000  0.000000   0.0  0.

In [11]:
#| code-summary: 'Display a few per-topic row term sums'
topic_model.c_tf_idf_[0:10].sum(axis=1)

matrix([[3.81681965],
        [3.56883265],
        [3.65906348],
        [3.62260767],
        [3.74989421],
        [3.62610859],
        [3.99024553],
        [3.54450792],
        [3.60094313],
        [3.7742639 ]])

## ✈️ Topics Summary

▶️ The `get_topic_info()` method is the best way to get a high-level summary of the topics that BERTopic generated.<br>
👁️ Inspect:

- For topic 0, find one of its **representative terms** in the topic 0 row in the **topic-term matrix** above. It should have a relatively high value.
- Are there representative words that do not help to interpret topics such as very common or rare words? Note any **need for vocabulary normalization** when we re-vectorize below.
  
🔄 View more top topics with `.head(top_n)`

📚 BERTopic [get_topic_info()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.get_topic_info), [set_topic_labels()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.set_topic_labels), [update_topics()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.update_topics)<br>
📝 Notice **topic -1**. HDBSCAN automatically identifies outliers which can not be clustered with any topic. BERTopic provides some strategies for [Outlier reduction](https://maartengr.github.io/BERTopic/getting_started/outlier_reduction/outlier_reduction.html).

In [22]:
#| code-summary: 'Display the topic summary by document count order'
#| code-fold: false
display(topic_model.get_topic_info().shape)
topic_model.get_topic_info().iloc[:, :-1].head(10) # MODIFY: to see more/less topics (omit Representative_Docs)

(40, 5)

,Topic,Count,Name,Representation
0,-1,705,-1_human_assistant_people_like,"[human, assistant, people, like, think, want, help, dont, know, im]"
1,0,116,0_recipe_potatoes_chicken_add,"[recipe, potatoes, chicken, add, dishes, sauce, vegetables, pepper, salt, cook]"
2,1,102,1_door_steal_money_bank,"[door, steal, money, bank, assistant, human, account, phone, tax, need]"
3,2,77,2_child_kids_baby_children,"[child, kids, baby, children, age, ideas, toys, assistant, human, gift]"
4,3,75,3_dog_cat_cats_animals,"[dog, cat, cats, animals, animal, dogs, pet, human, pets, assistant]"
5,4,53,4_hotel_visit_trip_tour,"[hotel, visit, trip, tour, mexico, sites, time, city, miami, day]"
6,5,50,5_human_assistant_im_sorry,"[human, assistant, im, sorry, rain, humans, say, understand, just, spider]"
7,6,50,6_video_blonde_job_hair,"[video, blonde, job, hair, women, assistant, human, networks, jobs, want]"
8,7,48,7_diet_weight_eat_eating,"[diet, weight, eat, eating, healthy, vitamin, health, foods, vegetables, meal]"
9,8,47,8_feel_friend_life_help,"[feel, friend, life, help, helpful, think, emotions, person, difficult, positive]"


## 🔎 Per-Topic Word Score
▶️ Below are two ways to analyze a specific topic(s)'s top word scores for more in-depth analysis: bar charts and raw scores.<br>
👁️ Do the word score distributions help make sense of some of the most distinctive/relevant topics? <br>
🔄 Enter the topic indices that you are interested in analyzing.

📚 BERTopic [Visualize Terms](https://maartengr.github.io/BERTopic/getting_started/visualization/visualize_terms.html), [visualize_barchart()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.visualize_barchart), [get_topic()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.get_topic)

In [23]:
#| code-summary: 'Visualize a list of topics by their word scores'
topic_model.visualize_barchart( # (defaults)
    topics = [0, 1, 2],         # (None), list overrides top_n_topics
    top_n_topics = 8,           # (8)
    n_words = 10,               # (5)
    autoscale = True,           # (False), may not see all words otherwise!
)

In [24]:
#| code-summary: 'Analyze one topic by its raw word scores'
topic_model.get_topic(0)

[('recipe', np.float64(0.03981477643456402)),
 ('potatoes', np.float64(0.037847837921660354)),
 ('chicken', np.float64(0.03241169530323849)),
 ('add', np.float64(0.030624310158523356)),
 ('dishes', np.float64(0.030277654229436023)),
 ('sauce', np.float64(0.02599058782634648)),
 ('vegetables', np.float64(0.025830420493995716)),
 ('pepper', np.float64(0.025568498188279384)),
 ('salt', np.float64(0.023448459210100803)),
 ('cook', np.float64(0.023276430200886034))]

## Change Topic Representation
We can change which words are used to represent the topic (steps #4-5) by using various tokenization/normalization options that we are already familiar with from Labs 2-3 using `update_topics()` such as:

- Removing stopwords recommended [here](https://maartengr.github.io/BERTopic/getting_started/tips_and_tricks/tips_and_tricks.html#removing-stop-words)
- Lemmatization ← **via our old friend spaCy!**
- N-grams
- Corpus-level vocabulary control: `max_df` (common words), `min_df` (rare words), `max_features`

🔄 Add your options to the `Count_Vectorizer()` configuration.<br>
▶️ Call `update_topics()` with the new vectorizer and include any other changes provided by its parameters.<br>
👁️ Does the vocabulary size change as expected? Does the new representation make the **topic more interpretable?**<br>

📚 scikit-learn [CountVectorizer()](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html), BERTopic [Update Topic Representations](https://maartengr.github.io/BERTopic/getting_started/topicrepresentation/topicrepresentation.html), [update_topics()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.update_topics)<br>
📝 Notes:

- This involves re-calculating the cTF-IDF weights based on the new representation.
- This does NOT change the actual topic clustering from step #3. Modularity! ✨

In [28]:
#| code-summary: 'Change topic label names'
#| code-fold: false
topn = 20 # MODIFY?

vectorizer_model = CountVectorizer(stop_words='english', min_df=2, max_df=0.5) # MODIFY: normalization options
topic_model.update_topics(docs, vectorizer_model=vectorizer_model, top_n_words=topn) # MODIFY: n_gram_range?
print(f'Modified vocabulary size: {len(topic_model.vectorizer_model.get_feature_names_out())}')
topic_model.get_topic_info().iloc[:, :-1].head()

Modified vocabulary size: 7132


,Topic,Count,Name,Representation
0,-1,705,-1_food_states_united_social,"[food, states, united, social, car, ai, computer, war, buy, email, government, medical, animals, goals, eat, humans, practice, house, snow, cause]"
1,0,116,0_recipe_potatoes_chicken_dishes,"[recipe, potatoes, chicken, dishes, sauce, pepper, vegetables, salt, cook, 12, meat, garlic, rice, potato, chopped, pasta, recipes, cup, boil, ingredients]"
2,1,102,1_door_steal_money_bank,"[door, steal, money, bank, account, phone, tax, credit, access, escape, hack, security, ssn, store, trunk, someones, card, image, victim, car]"
3,2,77,2_child_kids_baby_age,"[child, kids, baby, age, children, toys, gift, parents, books, school, kid, toy, rings, princess, homework, buy, mother, mouth, proud, discipline]"
4,3,75,3_dog_cat_cats_animals,"[dog, cat, cats, animals, dogs, animal, pet, pets, zoo, training, litter, puppy, breeds, box, forest, laws, wild, fetch, walk, reward]"


## Change Topic Naming

We can also change how the topics are named.

▶️ We use `set_topic_labels()` to manually assign new topic names. It can also be used with `generate_topic_labels()` to change the name formatting template from `'i_word1_word2_word3_word4'`.<br>
🔄 Update any topic names that would help with later analysis.

📚 BERTopic: [set_topic_labels()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.set_topic_labels), [generate_topic_labels()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.generate_topic_labels) (formatting parameters)

In [31]:
#| code-summary: 'Change topic label names'
#| code-fold: false
topic_model.set_topic_labels({0: 'Food', 1: 'Crime', 2: 'Children'}) # MODIFY: rename to better labels
topic_model.get_topic_info().iloc[:, :-2].head() # check after changes

,Topic,Count,Name,CustomName
0,-1,705,-1_food_states_united_social,-1_food_states_united_social
1,0,116,0_recipe_potatoes_chicken_dishes,Food
2,1,102,1_door_steal_money_bank,Crime
3,2,77,2_child_kids_baby_age,Children
4,3,75,3_dog_cat_cats_animals,3_dog_cat_cats_animals


## 🟣 🔵 🟢 Topic-Topic Relationships

Two useful visualizations can be used in conjunction to look for topic-to-topic relationships, including merging candidates.

▶️ The first is `visualize_topics()` where topics can be mapped to 2-dimensions using UMAP.<br>
👁️ Do some/many topics cluster together? Does the clustering make sense to you?<br>
🔄 Try mapping both embeddings and cTF-IDF weights by toggling the `use_ctfidf` parameter.

📚 BERTopic: [Visualize Topics](https://maartengr.github.io/BERTopic/getting_started/visualization/visualize_topics.html), [visualize_topics()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.visualize_topics)

In [32]:
#| code-summary: 'Visualize topics in UMAP 2-dimensional space'
topic_model.visualize_topics( # (defaults)
    use_ctfidf = False,       # (False) map embeddings / True: map cTF-IDF weights
    top_n_topics = None,      # (None) all
    custom_labels = True,     # (False)
)

The second visualization is even more useful to identify clusters of topics by high **cosine similarity**. 

▶️ Run `visualize_heatmap()` to get the idea of topic-to-topic simiarity.<br>
🔄 Modify:

- Recall the `visualize_topics()` 2d plot above to guess an initial `n_clusters` to cluster similar topics. **This is very useful and highly recommended** for analyzing potential topics to merge.
- Try different values of `n_clusters`
- Try both embeddings and cTF-IDF using `use_ctfidf`. Which one exhibits more distinctive clusters?

👁️ Inspect:

- Is it now clear from the similarity clustering which topics are candidates for merging?
- Or do you need to review some actual documents first? → Skip to **BERTopic Documents**.

📚 BERTopic: [Visualize Topics](https://maartengr.github.io/BERTopic/getting_started/visualization/visualize_topics.html), [visualize_heatmap()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.visualize_heatmap)

In [34]:
#| code-summary: "Visualize topic embeddings' cosine similarities and look for clusters of topics"
topic_model.visualize_heatmap( # (defaults)
    use_ctfidf = False,        # (False) map embeddings / True: map cTF-IDF weights
    n_clusters = 10,           # (None) <- MODIFY: powerful! creates ordered CLUSTERS of similar topics
    top_n_topics = None,       # (None) all
    custom_labels = True,      # (False)
)

# BERTopic Modify Topics
BERTopic provides an informative page on various [Topic Reduction](https://maartengr.github.io/BERTopic/getting_started/topicreduction/topicreduction.html) strategies.

Each modification below should be followed by running:

1. X1: Re-Check Topic Summary
2. And if effective: X2. Re-Check Topic Visualizations

## 🌟 Set `min_topic_size` (Step #3: HDBSCAN clustering)
This is the preferred method: to **rely on the HDBSCAN clustering algorithm** to find your number of topics. This can be used to **both decrease and increase** the number of topics that the clustering provides.

🔄 Check the topic summary to see what topic sizes (under **Count**) are being created, and modify `min_topic_size` accordingly.

- for less topics, try increasing `min_topic_size`
- for more topics, try decreasing `min_topic_size`

▶️ This needs to be done at BERTopic instantiation, i.e., the first cell here under **BERTopic Default Topics**<br>
👁️ Re-check the topic summary and see how the number of topics and their representations changed.<br>

📚 BERTopic: [Topic Reduction](https://maartengr.github.io/BERTopic/getting_started/topicreduction/topicreduction.html), [min_topic_size](https://maartengr.github.io/BERTopic/getting_started/parameter%20tuning/parametertuning.html#min_topic_size)

In [58]:
#| code-summary: 'Re-configure and train BERTopic, modifying `min_topic_size`'
#| code-fold: false

# Configure
topic_model = BERTopic(             # (defaults)
    # 3. Clustering
    min_topic_size = 10,            # (10) important! equivalent to HDBSCAN min_cluster_size
    
    # 4. Tokenizer/Vectorizer       # (CountVectorizer())
    vectorizer_model = CountVectorizer(stop_words="english", min_df=2),
    
    # 5. cTF-IDF -> Topic-Terms
    top_n_words = 10,               # (10) increase for more detailed topic descriptions
    nr_topics = None,               # (None) heed warnings!
)
# Train
topics, probs = topic_model.fit_transform(docs) # topics, probs may get updated later so we later use "live" attributes instead!
print(f'Modified number of topics (incl. -1): {topic_model.get_topic_info().shape[0]}')
print(f'Modified vocabulary size: {len(topic_model.vectorizer_model.get_feature_names_out())}\n')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1377.45it/s]


Modified number of topics (incl. -1): 41
Modified vocabulary size: 7377



## A. Reduce topics automatically (HDBSCAN on cTF-IDF vectors)

▶️ We can re-use HDBSCAN to automatically reduce topics by clustering the cTF-IDF vectors (post step #5).<br>
👁️ Re-check the topic summary and see how the number of topics and their new representations changed.<br>

📚 BERTopic: [Automatic Topic Reduction](https://maartengr.github.io/BERTopic/getting_started/topicreduction/topicreduction.html#automatic-topic-reduction), [reduce_topics(nr_topics='auto')](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.reduce_topics)<br>
📝 Notes:

- This may be a good first option if there are too many initial topics for manual merging
- `nr_topics='auto'` can also specified at BERTopic instantiation

In [59]:
#| code-summary: 'Run HDBSCAN to cluster topics by their cTF-IDF vectors'
#| code-fold: false
topic_model.reduce_topics(docs, nr_topics='auto') # can be run n times until converges
print(f'Modified number of topics (incl. -1): {topic_model.get_topic_info().shape[0]}')

Modified number of topics (incl. -1): 30


## B. Merge known topics (Post-Training on cTF-IDF vectors)

🔄 Create a list (or list of lists) to merge specific topics that you noted from your analysis above.<br>
▶️ Input the list to `merge_topics()`<br>
👁️ Re-check the topic summary and see how the merged topics were combined (check **Count**, **Name**, **Representation**), and all topics re-ordered.<br>

📚 BERTopic: [Manual Topic Reduction](https://maartengr.github.io/BERTopic/getting_started/topicreduction/topicreduction.html#manual-topic-reduction), [merge_topics()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.merge_topics)

In [60]:
#| code-summary: 'Merge topics that you specify'
#| code-fold: false
topics_to_merge = [1, 2] # MODIFY: list (or list of lists), should not re-run as numbers are re-arranged
topic_model.merge_topics(docs, topics_to_merge)
print(f'Modified number of topics (incl. -1): {topic_model.get_topic_info().shape[0]}')

Modified number of topics (incl. -1): 29


## <span style="color: red;">C. Reduce topics to fixed number (Post-Training on cTF-IDF vectors)</span>
🔄 Set a fixed number of topics in the `nr_topics` parameter<br>
▶️ Run `reduce_topics()` with this `nr_topics` number.<br>
👁️ Re-check the topic summary and see how the topics were reduced to the fixed number.<br>

📚 BERTopic: [Topic Reduction after Training](https://maartengr.github.io/BERTopic/getting_started/topicreduction/topicreduction.html#topic-reduction-after-training), [reduce_topics](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.reduce_topics)<br>
⚠️ Warning: `nr_topics` can force K topics post-clustering but is <span style="color: red;">warned against</span> [here](https://maartengr.github.io/BERTopic/getting_started/topicreduction/topicreduction.html) and <span style="color: red;">noted as "tricky"</span> [here](https://maartengr.github.io/BERTopic/getting_started/parameter%20tuning/parametertuning.html#nr_topics).

In [61]:
#| code-summary: 'Reduce topics to a specified number of topics'
#| code-fold: false
nr_topics = 10 # MODIFY: number of topics
topic_model.reduce_topics(docs, nr_topics=nr_topics)
print(f'Modified number of topics (incl. -1): {topic_model.get_topic_info().shape[0]}')

Modified number of topics (incl. -1): 10


## X1. Re-Check Topic Summary
👁️ Check modified topics after each modification step above.<br>
🔄 View more top topics with `.head(top_n)`

📚 BERTopic [get_topic_info()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.get_topic_info)

In [62]:
#| code-summary: 'Display the topic summary by document count order'
display(topic_model.get_topic_info().shape)
topic_model.get_topic_info().iloc[:, :-1].head(10) # omit Representative_Docs

(10, 5)

,Topic,Count,Name,Representation
0,-1,659,-1_human_assistant_people_like,"[human, assistant, people, like, think, want, im, dont, help, youre]"
1,0,573,0_human_assistant_make_like,"[human, assistant, make, like, use, want, sure, help, good, recipe]"
2,1,476,1_human_assistant_people_like,"[human, assistant, people, like, want, think, im, youre, help, just]"
3,2,68,2_game_games_video_play,"[game, games, video, play, assistant, human, players, points, ball, teams]"
4,3,60,3_human_film_assistant_movies,"[human, film, assistant, movies, story, movie, like, days, think, books]"
5,4,51,4_visit_hotel_assistant_human,"[visit, hotel, assistant, human, time, trip, tour, day, mexico, city]"
6,5,45,5_hair_want_blonde_human,"[hair, want, blonde, human, assistant, exercise, women, exercises, try, help]"
7,6,28,6_music_song_genre_10,"[music, song, genre, 10, favorite, musical, artists, human, assistant, band]"
8,7,24,7_address_locate_home_human,"[address, locate, home, human, assistant, trying, provide, chairman, congress, sure]"
9,8,16,8_symptoms_ear_kidney_kidneys,"[symptoms, ear, kidney, kidneys, doctor, infection, cancer, pain, cause, medical]"


### X2. Re-Check Visualizations
👁️ Check topic visualizations after each effective modification step above.<br>
🔄 Try both embeddings and cTF-IDF using `use_ctfidf`

📚 BERTopic [visualize_topics()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.visualize_topics), [visualize_heatmap()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.visualize_heatmap)

In [63]:
#| code-summary: 'Visualize topics in UMAP 2-dimensional space'
topic_model.visualize_topics(
    use_ctfidf = False, # default False (by embeddings) / True (by cTF-IDF)
    top_n_topics = None, # default None (all)
    custom_labels = True, # default False
)

In [64]:
#| code-summary: "Visualize topic embeddings' cosine similarities and look for clusters of topics"
topic_model.visualize_heatmap(
    use_ctfidf = False, # default False (by embeddings) / True (by cTF-IDF)
    #n_clusters = 8, # default None <- powerful! creates ordered CLUSTERS of similar topics
    top_n_topics = None, # default None
    custom_labels = True, # default False
)

# BERTopic Documents

## Document-Topic Matrix

▶️ Run `approximate_distribution()` to generate a "post-hoc approximation of topic distributions across documents" calculated on the cTF-IDF representations (step #5).<br>
👁️ Inspect:

- Notice the meaning of the dimensions.
- The topic row should all sum up to 1.0.

📚 BERTopic: [Topic Distributions](https://maartengr.github.io/BERTopic/getting_started/distribution/distribution.html), [approximate_distribution()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.approximate_distribution)<br>
⚠️ The outliers topics -1 is not included in the output of `approximate_distribution()`<br>
📝 The model's `probabilities_` attribute can similarly be analyzed and visualized. It represents "an approximation of topic probabilities as used in HDBSCAN (step #3)" and requires `calculate_probabilities = True` during [instantiation](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.__init__).

In [65]:
#| code-summary: "Display document-topic matrix as topic distributions"
topic_distr, _ = topic_model.approximate_distribution(docs, min_similarity=0)
display(topic_distr.shape)
pd.DataFrame(
    topic_distr, # can limit [:n]
    columns = topic_model.get_topic_info()['Topic'][1:] # distributions do not include topic -1!
).rename_axis('document_id').head()

(2000, 9)

Topic,0,1,2,3,4,5,6,7,8
document_id,,,,,,,,,
0,0.139031,0.187651,0.119272,0.125026,0.135775,0.073925,0.087147,0.082499,0.049674
1,0.171374,0.158782,0.112248,0.176821,0.111030,0.136675,0.052551,0.040515,0.040003
2,0.165915,0.182560,0.091715,0.086991,0.118817,0.081524,0.080560,0.107229,0.084690
3,0.144105,0.201498,0.104063,0.115935,0.093576,0.109267,0.094129,0.076050,0.061377
4,0.112280,0.144668,0.087519,0.155823,0.095779,0.083451,0.224673,0.044171,0.051636


In [66]:
#| code-summary: 'Display a few per-document row topic sums'
topic_distr[0:10].sum(axis=1)[:, np.newaxis]

array([[1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.]])

## ✈️ Documents Summary

▶️ The `get_document_info()` method summarizes each document's topic assignment and probability.<br>
👁️ For any document, compare the **Probability** score with the topic **distribution** for that document in the document-topic matrix above.<br>
🔄 View more documents with `.head(top_n)` or `.sample(n)`

📚 BERTopic: [get_document_info()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.get_document_info)

In [67]:
#| code-summary: 'Display the document summary by document number order'
topic_model.get_document_info(docs)[['Topic', 'Name', 'Top_n_words', 'Probability']].head()

,Topic,Name,Top_n_words,Probability
0,-1,-1_human_assistant_people_like,human - assistant - people - like - think - want - im - dont - help - youre,0.000000
1,0,0_human_assistant_make_like,human - assistant - make - like - use - want - sure - help - good - recipe,0.819750
2,1,1_human_assistant_people_like,human - assistant - people - like - want - think - im - youre - help - just,0.944089
3,-1,-1_human_assistant_people_like,human - assistant - people - like - think - want - im - dont - help - youre,0.000000
4,6,6_music_song_genre_10,music - song - genre - 10 - favorite - musical - artists - human - assistant - band,1.000000


## 🔎 Per-Document Topic Distribution

🔄 Set the document index in the two cells below to visualize and read a specific document.<br>
▶️ `visualize_distribution()` shows one document's topic distribution as a bar chart.<br>
👁️ Does this topic distribution make sense for this document?<br>

📚 BERTopic: [Visualize Probabilities or Distribution](https://maartengr.github.io/BERTopic/getting_started/visualization/visualize_documents.html#visualize-probabilities-or-distribution), [visualize_distribution()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.visualize_distribution)

In [68]:
#| code-summary: "Visualize ONE document's topic distribution"
topic_model.visualize_distribution(                # (default)
    probabilities = topic_distr[1],                # MODIFY: document index number
    min_probability = 0.015,                       # (0.015)
    custom_labels = True,                          # (False)
)

In [69]:
#| code-summary: 'Read the actual document for validation'
docs[1] # MODIFY: document index number

'\n\nHuman: My daughter wants to dress up as a princess for her birthday. What are some good princess costumes that I can buy?\n\nAssistant: What kind of princess does she want to be?  Is she interested in a specific movie princess, or in a generic princess?  Does she want to be a specific princess from history, or just a generic princess?'

## 🟣 🔵 🟢 Document-Topic Relationships

▶️ `visualize_documents()` displays all of the documents using their original (step #1) embeddings reduced to 2-dimensional space and colored by topic assignment<br>
👁️ Hover over the documents to see their text. Are the topics recognizable when reading each document?<br>

📚 BERTopic: [Visualize documents with Plotly](https://maartengr.github.io/BERTopic/getting_started/visualization/visualize_documents.html#visualize-documents-with-plotly), [visualize_documents()](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.visualize_documents)<br>
📝 As suggested in the documentation, we pre-reduce the embeddings from `SentenceTransformer` by directly calling `UMAP`.

In [70]:
#| code-summary: 'Visualize documents in 2D space'
from sentence_transformers import SentenceTransformer
from umap import UMAP

sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = sentence_model.encode(docs, show_progress_bar=False) # docs should be a list (not Pandas Series)!
reduced_embeddings = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine').fit_transform(embeddings)
topic_model.visualize_documents(docs, reduced_embeddings=reduced_embeddings) # docs should be a list (not Pandas Series)!

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7872.54it/s]


# Topic Modelling Evaluation Metrics

The evaluation of the quality of the resulting topics from topic modelling is inherently subjective and requires **human oversight**. There are however several commonly used metrics that can be part of your evaluation strategy together with human interpretation. A good overview is provided by [Abdelrazek et al. (2023)](https://www.sciencedirect.com/science/article/abs/pii/S0306437922001090). We look at **coherence** and **diversity** here, which were used in the BERTopic [paper](https://arxiv.org/abs/2203.05794) to compare performance against other topic modelling approaches and variations of embedding models within its own flexible framework.

**Coherence** aims to measure semantic **similarity WITHIN** each topic by calculating how often the top N terms in a topic (as word pairs) co-occur in the corpus. See papers by [Newman et al. (2010)](https://aclanthology.org/N10-1012.pdf) and [Syed & Spruit (2017)](https://ieeexplore.ieee.org/abstract/document/8259775).
- Coherence requires each topic's **top N words** and the **corpus** to check for co-occurrence of word pairs along a sliding window of variable size.
- We use **Gensim's** [CoherenceModel](https://radimrehurek.com/gensim/models/coherencemodel.html) implementation.
- There are several measures of coherence. Here, we use Normalized Pointwise Mutual Information (NPMI) by default which varies in the range of [-1, 1]

**Diversity** aims to measure each topic's **uniqueness COMPARED TO** other topics by calculating the percentage of unique words in the top N words of each topic, and is briefly desribed by [Dieng et al. (2020)](https://direct.mit.edu/tacl/article/doi/10.1162/tacl_a_00325/96463/Topic-Modeling-in-Embedding-Spaces).
- Diversity only requires each topic's **top N words** and varies from [0, 1]
- We define a simple function to calculate diversity directly copying code from the [TopicDiversity](https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/diversity_metrics.py) class of the **[OCTIS](https://github.com/MIND-Lab/OCTIS)** package.

In [ ]:
#| code-summary: 'Define evaluation functions to measure coherence and diversity'
def get_diversity_octis(topics, topn=10):
    '''Calculate topic diversity. Code ref: https://github.com/MIND-Lab/OCTIS.'''
    unique_words = set()
    for topic in topics:
        unique_words = unique_words.union(set(topic[:topn]))
    return len(unique_words) / (topn * len(topics))

def evaluate_bertopic(model_string, bertopic_model, analyzer, docs, window_size=None, topn=None, coherence_measure='c_npmi'):
    '''Calculate coherence and diversity for a BERTopic model.'''

    # Get BERTopic topics-terms (topn)
    if topn is None:
        topn = len(topic_model.get_topic(0))
    topics_topterms = [
        [word for word, score in bertopic_model.get_topic(topic_id)[:topn]]
         for topic_id in bertopic_model.get_topics()
         if topic_id != -1
    ]

    # Coherence needs docs tokenized in original order
    texts_as_tokens = [analyzer(doc) for doc in docs]
    dictionary = corpora.Dictionary(texts_as_tokens)

    tc = CoherenceModel(
        topics=topics_topterms,
        texts=texts_as_tokens, dictionary=dictionary,
        window_size=window_size, topn=topn, coherence=coherence_measure
    ).get_coherence()
    
    td = get_diversity_octis(topics_topterms, topn)

    return {'model': model_string, 'k_topics': len(topics_topterms), 'coherence': tc, 'diversity': td}

def evaluate_ldia(model_string, df_ldia_topics_terms, analyzer, docs, window_size=None, topn=None, coherence_measure='c_npmi'):
    '''Calculate coherence and diversity for a Latent Dirichelet Allocation model.'''

    # Get LDiA topics-terms (topn)
    if topn is None:
        topn = len(df_ldia_topics_topterms)
    topics_topterms = []
    topics_i = df_ldia_topics_topterms.columns.get_level_values(0).unique()
    for topic_i in topics_i:
        topics_topterms.append(df_ldia_topics_topterms[topic_i]['term'].tolist())

    # Coherence needs docs tokenized in original order
    texts_as_tokens = [analyzer(doc) for doc in docs]
    dictionary = corpora.Dictionary(texts_as_tokens)
    
    tc = CoherenceModel(
        topics=topics_topterms,
        texts=texts_as_tokens, dictionary=dictionary,
        window_size=window_size, topn=topn, coherence=coherence_measure
    ).get_coherence()
    
    td = get_diversity_octis(topics_topterms, topn)

    return {'model': model_string, 'k_topics': len(topics_topterms), 'coherence': tc, 'diversity': td}

In [ ]:
#| code-summary: 'Define function: create DataFrame of top constituent term/topic columns for each topic/document row'
def create_df_rows_topcols(row_col_matrix, rows_i, cols_i, col_type, topcol_n):
    '''Finds the top scoring terms (columns) from a Topic-Term matrix, and wraps into a DataFrame.'''
    list_df_rows = []
    for idx, row in enumerate(row_col_matrix):
        row_scores = [{col_type: cols_i[i], 'score': row[i]} for i in row.argsort()[:-topcol_n - 1:-1]]
        df_row = pd.DataFrame(row_scores)
        df_row.columns = pd.MultiIndex.from_product([[rows_i[idx]], df_row.columns])
        list_df_rows.append(df_row)

    return pd.concat(list_df_rows, axis=1)

## Compare with an LDiA Topic Model

🔄 Set `k_topics_ldia` for a fair comparison with BERTopic. Should you use the same K or use the optimized K for both?<br>
▶️ The code below condenses the pipeline from Lab 6 for LDiA.<br>
👁️ How do these topics compare to those from BERTopic?<br>

📚 scikit-learn [LatentDirichletAllocation](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.LatentDirichletAllocation.html)<br>
📝 Note that we reuse the CountVectorizer from our BERTopic pipeline for a consistent comparison.

In [ ]:
#| code-summary: 'Run the LDiA flow, condensed from Lab 6'
k_topics_ldia = 14 # MODIFY: vs. BERTopic K topics, equalize or optimize each?

# Topic modelling
X_bow = topic_model.vectorizer_model.transform(docs)
ldia_model = LatentDirichletAllocation(n_components=k_topics_ldia)
ldia_model.fit_transform(X_bow)

# Data wrangling to view top terms per topic
ldia_topics = ldia_model.components_
ldia_topics_norm = ldia_topics / ldia_topics.sum(axis=1)[:, None]
ldia_topics_i = [f'topic_{i}' for i in range(ldia_model.n_components)]
ldia_terms = topic_model.vectorizer_model.get_feature_names_out()
df_ldia_topics_topterms = create_df_rows_topcols(ldia_topics_norm, rows_i=ldia_topics_i, cols_i=ldia_terms, col_type='term', topcol_n=topn)
df_ldia_topics_topterms

## Evaluate: Compare Metrics

▶️ The code below collects evaluation metrics for each model for comparison.<br>
👁️ Are the results surprising or consistent with your expectations?<br>

📝 Notes:

- For a project evaluation, it is common to run each model several times as both BERTopic and LDiA are non-deterministic in nature.
- Here, be aware that we are measuring the coherence of the topics based on the corpus that was used to generate them, a sort of **in-sample check**. If our goal is for the topics to be **generalizable**, then you can run coherence on held-out test data.
- We reuse the same CountVectorizer's configured `build_analyzer()` to rebuild the corpus as tokenized texts, required to calculate coherence.

In [ ]:
#| code-summary: 'Evaluate and compare models'
#| code-fold: false
analyzer = topic_model.vectorizer_model.build_analyzer() # Equalize preprocessing
results = list()
results.append(evaluate_bertopic('BERTopic', topic_model, analyzer, docs))
results.append(evaluate_ldia('LDiA', df_ldia_topics_topterms, analyzer, docs))
pd.DataFrame(results).set_index('model').round(4)